# IBM Quantum / Qiskit — Six Circuit Exercises

This notebook builds the six required circuits, gives predictions, and checks the unitary equivalence for Questions 3 and 6.

**Qiskit bit order:** count strings are shown as `q[n-1] ... q[0]`. Therefore a physical state with `q0=0, q1=1` is displayed as `10`.

## Setup

We only need to do this once. Run the cell below before anything else in this notebook.

- `%pip install ...` makes sure the needed packages are present.
- `import numpy as np` gives us NumPy, which we already used in the first notebook.
- `from qiskit import QuantumCircuit` gives us the tool for building a circuit.
- `from qiskit.quantum_info import Statevector, Operator` gives us four tools we will use throughout this notebook: `Statevector` for a fully known state, `Operator` for a gate's matrix.
- `from qiskit.visualization import plot_bloch_multivector, plot_state_city, plot_histogram` gives us three ways to draw a state as a picture. `plot_histogram` is new here, it draws a set of probabilities as bars, one bar per possible outcome.

We will not need a simulator or shots in this notebook. `Statevector` already tells us the exact outcome and the exact probabilities directly, with no need to run a circuit many times and count results. The simulator becomes useful later in the course, once we reach algorithms where we genuinely want to watch randomness play out over many runs.

In [9]:
%pip install qiskit matplotlib numpy
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.visualization import plot_bloch_multivector, plot_histogram

In [10]:
%pip install qiskit-aer

In [11]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
try:
    from qiskit_aer import AerSimulator
except ImportError:
    AerSimulator = None

def run_counts(circuit, shots=4096):
    """Run measurement circuit locally if qiskit-aer is installed."""
    if AerSimulator is None:
        print("Install qiskit-aer to simulate counts: pip install qiskit-aer")
        return None
    result = AerSimulator().run(circuit, shots=shots).result()
    counts = result.get_counts()
    print(counts)
    return counts


## Question 1 — Build \(|01\rangle\)

Apply one X gate to `q1`. Prediction: `q0=0, q1=1`; Qiskit prints this as `10`.

In [12]:
q1 = QuantumCircuit(2, 2)
q1.x(1)
q1.measure([0, 1], [0, 1])
print(q1.draw())
run_counts(q1)

          ┌─┐   
q_0: ─────┤M├───
     ┌───┐└╥┘┌─┐
q_1: ┤ X ├─╫─┤M├
     └───┘ ║ └╥┘
c: 2/══════╩══╩═
           0  1 
{'10': 4096}


{'10': 4096}

I applied X to q1. The result was q0=0 and q1=1; IBM displayed 10 due to Qiskit bit ordering

## Question 2 — Toffoli gate prediction

Common setup: X on both controls (`q0`, `q1`), then `CCX(q0, q1, q2)`. Prediction: all qubits end as 1, so the result is `111`.

In [13]:
q2 = QuantumCircuit(3, 3)
q2.x(0)
q2.x(1)
q2.ccx(0, 1, 2)
q2.measure([0, 1, 2], [0, 1, 2])
print(q2.draw())
run_counts(q2)

     ┌───┐     ┌─┐      
q_0: ┤ X ├──■──┤M├──────
     ├───┤  │  └╥┘┌─┐   
q_1: ┤ X ├──■───╫─┤M├───
     └───┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ─────┤ X ├─╫──╫─┤M├
          └───┘ ║  ║ └╥┘
c: 3/═══════════╩══╩══╩═
                0  1  2 
{'111': 4096}


{'111': 4096}

## Question 3 — CNOT from H and CZ only

Use \(H\) on the target, then CZ, then \(H\) on the target: \(CX_{0\to1}=H_1 CZ_{0,1} H_1\).

In [14]:
q3 = QuantumCircuit(2)
q3.h(1)
q3.cz(0, 1)
q3.h(1)

cnot_reference = QuantumCircuit(2)
cnot_reference.cx(0, 1)

print(q3.draw())
print("Matches CNOT:", Operator(q3).equiv(Operator(cnot_reference)))
print("Built matrix:\n", Operator(q3).data)

                  
q_0: ──────■──────
     ┌───┐ │ ┌───┐
q_1: ┤ H ├─■─┤ H ├
     └───┘   └───┘
Matches CNOT: True
Built matrix:
 [[1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]]


I used H(q1) → CZ(q0,q1) → H(q1). The unitary equivalence check with CNOT returned True

## Question 4 — CNOT on \(|11\rangle\)

Prepare both qubits as 1 and apply `CX(0, 1)`. Prediction: the target flips from 1 to 0. Thus `q0=1, q1=0`; Qiskit prints `01`.

In [15]:
q4 = QuantumCircuit(2, 2)
q4.x(0)
q4.x(1)
q4.cx(0, 1)
q4.measure([0, 1], [0, 1])
print(q4.draw())
run_counts(q4)

     ┌───┐     ┌─┐   
q_0: ┤ X ├──■──┤M├───
     ├───┤┌─┴─┐└╥┘┌─┐
q_1: ┤ X ├┤ X ├─╫─┤M├
     └───┘└───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 
{'01': 4096}


{'01': 4096}

## Question 5 — Two independent Hadamards

Apply H to both qubits. Prediction: `00`, `01`, `10`, and `11` each occur at approximately 25%.

In [16]:
q5 = QuantumCircuit(2, 2)
q5.h(0)
q5.h(1)
q5.measure([0, 1], [0, 1])
print(q5.draw())
run_counts(q5)

     ┌───┐┌─┐   
q_0: ┤ H ├┤M├───
     ├───┤└╥┘┌─┐
q_1: ┤ H ├─╫─┤M├
     └───┘ ║ └╥┘
c: 2/══════╩══╩═
           0  1 
{'11': 1015, '01': 995, '00': 1053, '10': 1033}


{'11': 1015, '01': 995, '00': 1053, '10': 1033}

## Question 6 — SWAP from CNOTs only

Use `CX(0,1)`, `CX(1,0)`, and `CX(0,1)`.

In [17]:
q6 = QuantumCircuit(2)
q6.cx(0, 1)
q6.cx(1, 0)
q6.cx(0, 1)

swap_reference = QuantumCircuit(2)
swap_reference.swap(0, 1)

print(q6.draw())
print("Matches SWAP:", Operator(q6).equiv(Operator(swap_reference)))
print("Built matrix:\n", Operator(q6).data)

          ┌───┐     
q_0: ──■──┤ X ├──■──
     ┌─┴─┐└─┬─┘┌─┴─┐
q_1: ┤ X ├──■──┤ X ├
     └───┘     └───┘
Matches SWAP: True
Built matrix:
 [[1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]]


I used CX(0,1) → CX(1,0) → CX(0,1). The unitary equivalence check with SWAP returned True